# MetroPT-3 — Failure-Window Identification (EDA)

**Purpose:** help a HUMAN identify the real failure date-ranges by looking at the signals. This notebook does **not** decide failures algorithmically — using an anomaly detector to label the data we'll later train an anomaly detector on would be circular. Every plot here is descriptive; the failure windows at the bottom are a judgment call the user records with reasoning, then cross-checks against the dataset documentation (`Data Description_Metro.pdf`, from the UCI download).

Read-only: nothing here writes to `data/`, and no cell mutates the loaded DataFrame in place (downsampling for a plot always goes to a separate local variable).

## 1. Setup — load config and the raw data (reusing the package, not re-implementing it)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from apu_sentinel.config import load_config
from apu_sentinel.data.load import load_raw

settings = load_config("local")  # CONFIG=colab for the full-data environment

raw_path = Path(settings.data.raw_dir) / settings.data.raw_filename
df = load_raw(raw_path)

print("shape:", df.shape)
print("columns:", list(df.columns))
print("time span:", df.index.min(), "->", df.index.max())

## 2. Signal inventory

MetroPT-3 records an Air Production Unit (compressor) at 0.1 Hz (one reading per ~10s). Per the dataset documentation, the columns split into:

**Analog (continuous, physically meaningful — the main channels for spotting anomalies):**
- `TP2` — pressure at the compressor
- `TP3` — pressure at the pneumatic panel / downstream reservoir
- `H1` — pressure downstream of the valve (drop across it with TP3)
- `DV_pressure` — pressure drop induced by the towers' valves during air drying
- `Reservoirs` — pressure at the reservoirs, downstream
- `Oil_temperature` — compressor oil temperature
- `Motor_current` — compressor motor current draw

**Digital / status (binary-ish, define the operating regime — CLAUDE.md's 'condition on regime' concern):**
- `COMP` — compressor electrical signal (on/off)
- `DV_eletric` — solenoid (air intake) valve electric signal
- `Towers` — which of the two drying towers is active
- `MPG` — starts the compressor when pressure drops below a threshold
- `LPS` — low-pressure switch (triggers below ~7 bar)
- `Pressure_switch` — discrete pressure switch state
- `Oil_level` — oil level indicator (low/ok)
- `Caudal_impulses` — counts airflow pulses

`Unnamed: 0` is a leftover row-index artifact from the source CSV, not a sensor — ignore it for signal analysis.

## 3. Full-timeline overview — each analog channel across the whole span

Resampled to 5-minute means **for this plot only** — `df` itself is untouched. Look for where a channel's level, noise, or drift departs from its months-long normal pattern; shared x-axis so a departure that lines up across several channels together is a stronger candidate than one that's isolated to a single channel.

In [ ]:
ANALOG_CHANNELS = [
    "TP2",
    "TP3",
    "H1",
    "DV_pressure",
    "Reservoirs",
    "Oil_temperature",
    "Motor_current",
]


# Downsampled copy for plotting only -- df is never mutated in place.
overview = df[ANALOG_CHANNELS].resample("5min").mean()

fig, axes = plt.subplots(
    len(ANALOG_CHANNELS), 1, figsize=(16, 2.2 * len(ANALOG_CHANNELS)), sharex=True
)
for ax, col in zip(axes, ANALOG_CHANNELS, strict=True):
    ax.plot(overview.index, overview[col], linewidth=0.6)
    ax.set_ylabel(col, rotation=0, ha="right", fontsize=9)
fig.suptitle("Analog channels, full time span (5min mean, plotting-only downsample)")
fig.tight_layout()
plt.show()

## 4. Operating-regime visibility — the compressor's on/off cycling

Most of the raw variance in the analog channels above is the compressor cycling on and off, not anomalous behaviour (CLAUDE.md hard rule 4). Plotting the digital/status signals here just makes that cycling VISIBLE — no regime segmentation or modeling happens in this notebook, that's a later pass.

In [ ]:
DIGITAL_CHANNELS = [
    "COMP",
    "DV_eletric",
    "Towers",
    "MPG",
    "LPS",
    "Pressure_switch",
    "Oil_level",
    "Caudal_impulses",
]


# Downsampled copy for plotting only -- df is never mutated in place.
regime_view = df[DIGITAL_CHANNELS].resample("5min").mean()  # fraction of time 'on' per bin

fig, axes = plt.subplots(
    len(DIGITAL_CHANNELS), 1, figsize=(16, 1.6 * len(DIGITAL_CHANNELS)), sharex=True
)
for ax, col in zip(axes, DIGITAL_CHANNELS, strict=True):
    ax.plot(regime_view.index, regime_view[col], linewidth=0.6)
    ax.set_ylabel(col, rotation=0, ha="right", fontsize=9)
fig.suptitle("Digital/status channels, full time span (5min mean = duty cycle)")
fig.tight_layout()
plt.show()

## 5. Zoom-in scaffolding

Reusable helper to zoom into a candidate window at full (10s) resolution across all key channels together. **Edit `EXAMPLE_START` / `EXAMPLE_END` below** to the date ranges you actually want to inspect — the values here are just a placeholder so the notebook runs top-to-bottom; they are NOT a claimed failure window.

In [ ]:
def plot_window(df, start, end, channels=None):
    """Plot `channels` (default: analog channels) over df.loc[start:end] at full
    resolution. Read-only -- `window` below is a slice/copy, df is untouched.
    """
    channels = channels or ANALOG_CHANNELS
    window = df.loc[start:end, channels]
    fig, axes = plt.subplots(len(channels), 1, figsize=(16, 2.2 * len(channels)), sharex=True)
    for ax, col in zip(axes, channels, strict=True):
        ax.plot(window.index, window[col], linewidth=0.8)
        ax.set_ylabel(col, rotation=0, ha="right", fontsize=9)
    fig.suptitle(f"{start} -> {end}")
    fig.tight_layout()
    return fig


EXAMPLE_START = "2020-06-05"  # <-- EDIT ME: candidate window start
EXAMPLE_END = "2020-06-08"  # <-- EDIT ME: candidate window end

_ = plot_window(df, EXAMPLE_START, EXAMPLE_END)
plt.show()

# You can also zoom on the digital channels for the same window, e.g.:
# _ = plot_window(df, EXAMPLE_START, EXAMPLE_END, channels=DIGITAL_CHANNELS)

## 6. Distributions / summary stats — characterising "normal"

Descriptive only -- summary stats and histograms of the analog channels over the whole series, to build intuition for what 'normal' looks like before judging what departs from it.

In [ ]:
df[ANALOG_CHANNELS].describe()

In [ ]:
fig, axes = plt.subplots(1, len(ANALOG_CHANNELS), figsize=(4 * len(ANALOG_CHANNELS), 3))
for ax, col in zip(axes, ANALOG_CHANNELS, strict=True):
    ax.hist(df[col].dropna(), bins=50)
    ax.set_title(col, fontsize=9)
fig.tight_layout()
plt.show()

## 7. Failure-window recording — YOUR judgment (the deliverable for the split pass)

Fill in `failure_windows` below with the periods **you** identified by eye in sections 3-5 above. For each one, record:
- `start` / `end` — the date-range, as timestamps
- `notes` — WHICH signals looked abnormal, WHAT pattern you saw, and confirmation you   cross-checked it against the dataset documentation (`Data Description_Metro.pdf`)

**Do not auto-generate this list.** No algorithm in this notebook decided these — that would be circular (labelling the training data for an anomaly detector using an anomaly detector). This is the label-construction step CLAUDE.md assigns to the user; it's the input the next (split) pass depends on.

In [ ]:
# TEMPLATE -- replace with your own visually-identified + doc-cross-checked windows.
# Each entry: (start, end, notes). Leave empty until you've done the visual inspection above.
failure_windows = [
    # {
    #     "start": "YYYY-MM-DDTHH:MM:SS",
    #     "end": "YYYY-MM-DDTHH:MM:SS",
    #     # which signals, what pattern, and doc cross-check go here:
    #     "notes": "e.g. TP2/Motor_current drifted starting ~X; matches doc p.N",
    # },
]
failure_windows

In [ ]:
print(len(df.columns), sorted(df.columns))

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

import apu_sentinel
from apu_sentinel.config import load_config
from apu_sentinel.data.load import load_raw
from apu_sentinel.data.scaling import fit_scaler
from apu_sentinel.data.split import apply_fold, make_folds

REPO_ROOT = Path(apu_sentinel.__file__).resolve().parents[2]

settings = load_config("local")
df = load_raw(REPO_ROOT / settings.data.raw_dir / settings.data.raw_filename)

folds = make_folds(settings, df.index.min(), df.index.max())
train_df, _ = apply_fold(df, folds[0])  # fold 1
scaler = fit_scaler(train_df, settings)
print(vars(scaler))  # what it actually learned

analog = list(settings.scaling.analog_columns)


def robust_stats(frame):
    q = frame[analog].quantile([0.25, 0.5, 0.75])
    return q.loc[0.5], q.loc[0.75] - q.loc[0.25]  # center, scale


f_center, f_scale = robust_stats(train_df)  # fold 1 clean train
a_center, a_scale = robust_stats(df)  # full series

print(
    pd.DataFrame(
        {
            "fold1_center": f_center,
            "full_center": a_center,
            "center_%diff": 100 * (a_center - f_center).abs() / f_center.abs(),
            "fold1_scale": f_scale,
            "full_scale": a_scale,
            "scale_%diff": 100 * (a_scale - f_scale).abs() / f_scale.abs(),
        }
    ).round(3)
)

In [ ]:
deltas = df.index.to_series().diff().dropna()
print(deltas.value_counts().head(10))
print("gaps > 1 min:", (deltas > pd.Timedelta("1min")).sum())

In [ ]:
gaps = deltas[deltas > pd.Timedelta("1min")].sort_values(ascending=False)
print(gaps.head(15))  # largest gaps and where they land
print("total missing:", gaps.sum())

for ev in settings.evaluation.failure_events:
    start = pd.Timestamp(ev.start)
    near = gaps[(gaps.index > start - pd.Timedelta("5D")) & (gaps.index < start)]

In [ ]:
import pandas as pd

# --- isolate OFF samples (COMP=1 per the verified polarity) ---
off = df[df["COMP"] == 1]

# --- the two modes inside OFF ---
low = off[off["Motor_current"] < 1.0]  # true off
high = off[off["Motor_current"] > 3.0]  # sustained ~3.5-4.3A
print(
    f"OFF total {len(off):,}  |  low {len(low):,} ({len(low) / len(off):.1%})"
    f"  |  high {len(high):,} ({len(high) / len(off):.1%})"
)

# --- does any remaining flag separate them? ---
candidates = [
    "Towers",
    "Pressure_switch",
    "LPS",
    "Caudal_impulses",
    "Oil_level",
    "DV_eletric",
    "MPG",
]

rows = []
for flag in candidates:
    if flag not in df.columns:
        continue
    p_low, p_high = low[flag].mean(), high[flag].mean()
    rows.append(
        {
            "flag": flag,
            "P(flag=1 | low)": round(p_low, 3),
            "P(flag=1 | high)": round(p_high, 3),
            "separation": round(abs(p_high - p_low), 3),
        }
    )

print(pd.DataFrame(rows).sort_values("separation", ascending=False).to_string(index=False))

In [ ]:
off = df[df["COMP"] == 1]
low = off[off["Motor_current"] < 1.0]  # stopped
high = off[off["Motor_current"] > 3.0]  # offload

analog = ["TP2", "TP3", "H1", "DV_pressure", "Reservoirs", "Oil_temperature", "Motor_current"]


def q(f, c):
    return f[c].quantile(0.75) - f[c].quantile(0.25)


comp = pd.DataFrame(
    {
        "stopped_med": low[analog].median(),
        "offload_med": high[analog].median(),
        "stopped_iqr": q(low, analog),
        "offload_iqr": q(high, analog),
    }
)
comp["med_shift"] = (comp.offload_med - comp.stopped_med).abs()
print(comp.round(4))

# is the threshold robust? how empty is the valley between modes?
print(f"\nOFF samples in the 1-3A valley: {off['Motor_current'].between(1.0, 3.0).mean():.3%}")

In [ ]:
# STOPPED-run duration over time — does it shorten before failures?
off = df[df["COMP"] == 1]
stopped = off[off["Motor_current"] < 1.0]
runs = (stopped.index.to_series().diff() > pd.Timedelta("60s")).cumsum()
dur = stopped.groupby(runs).apply(lambda g: (g.index[-1] - g.index[0]).total_seconds())
t = stopped.groupby(runs).apply(lambda g: g.index[0])

s = pd.Series(dur.values, index=t.values).sort_index()
ax = (
    s.rolling(20).median().plot(figsize=(14, 4), title="Median STOPPED duration (20-cycle rolling)")
)
for ev in settings.evaluation.failure_events:
    ax.axvline(pd.Timestamp(ev.start), color="red", ls="--")